In [2]:
# Step 1: Import Libraries 
import numpy as np
import pandas as pd
# !pip install yfinance
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import GRU, Dense 
from tensorflow.keras.optimizers import Adam
import datetime


In [5]:
# Step 2: Download Stock Data (10 years till yesterday)
ticker = "AAPL" 
end_date = datetime.date.today() - datetime.timedelta(days=1) 
start_date = end_date - datetime.timedelta(days=365*10)

In [ ]:
df = yf.download(ticker, start=start_date, end=end_date)
print(df.head())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2016-01-14  22.438671  22.655123  21.586399  22.086940  252680400
2016-01-15  21.899805  22.030578  21.500726  21.690119  319335600
2016-01-19  21.793829  22.242512  21.532284  22.188400  212350800
2016-01-20  21.823139  22.138796  21.063308  21.442096  289337600
2016-01-21  21.712658  22.068898  21.406020  21.884013  208646000


In [7]:
# Use multiple  features
features = ['Open', 'High', 'Low', 'Close', 'Volume'] 
data = df[features].values
print(data[:5]) 
# Print first 5 rows of data

[[2.20869401e+01 2.26551228e+01 2.15863987e+01 2.24386711e+01
  2.52680400e+08]
 [2.16901188e+01 2.20305775e+01 2.15007257e+01 2.18998051e+01
  3.19335600e+08]
 [2.21883996e+01 2.22425117e+01 2.15322842e+01 2.17938290e+01
  2.12350800e+08]
 [2.14420961e+01 2.21387961e+01 2.10633082e+01 2.18231392e+01
  2.89337600e+08]
 [2.18840131e+01 2.20688975e+01 2.14060201e+01 2.17126579e+01
  2.08646000e+08]]


In [8]:
# Step 3: Scale Data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

In [9]:
# Step 4: Create Dataset function
# 
def create_dataset(data, time_step=60): 
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step)]) # last 'time_step' days of all features
        y.append(data[i + time_step, [0, 3]])  # predict [Open, Close]
        return np.array(X), np.array(y)
      

In [10]:
time_step = 60
X, y = create_dataset(scaled_data, time_step)

In [11]:
# Reshape into 3D: [samples, time_steps, features] 
X = X.reshape(X.shape[0], X.shape[1], X.shape[2])

In [12]:
# Step 5: Build GRU Model
model = Sequential() 
model.add(GRU(units=64, return_sequences=True, input_shape=(time_step, X.shape[2])))
model.add(GRU(units=64)) 
model.add(Dense(units=2)) # Output both Open & Close 
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

d:\GENAI\DeepLearning\dlenv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Step 6: Train Model
model.fit(X, y, epochs=10, batch_size=32, verbose=0)
# # Step 7: Predict Today's Open & Close 
last_60_days = scaled_data[-time_step:] # last 60 rows #
print("Last 60 days data:", last_60_days) 
last_60_days = last_60_days.reshape(1, time_step, X.shape[2]) # reshape to 3D and include all features and batch size 1 for prediction print("Reshaped last 60 days data:", last_60_days) predicted_prices = model.predict(last_60_days)

Last 60 days data: [[0.8609134  0.86162825 0.86279793 0.86034096 0.03100075]
 [0.85625052 0.85125398 0.85390566 0.85323151 0.04241224]
 [0.8553856  0.8674497  0.86203788 0.87143766 0.06058636]
 [0.88497975 0.90849877 0.89380696 0.90886573 0.14076198]
 [0.90750439 0.91189464 0.91736765 0.91085936 0.05583219]
 [0.91039982 0.9027892  0.89304686 0.89460927 0.05257248]
 [0.90020923 0.89446738 0.90285123 0.89885978 0.02879212]
 [0.90490979 0.90756589 0.90729741 0.91104758 0.03945763]
 [0.91878547 0.92618709 0.92808399 0.9335795  0.05232596]
 [0.93424056 0.9290606  0.94138444 0.93429422 0.04582168]
 [0.93533112 0.93473281 0.9374323  0.93692742 0.06434862]
 [0.94552169 0.94492048 0.94263856 0.94332209 0.10081285]
 [0.9643236  0.95678739 0.94522266 0.93944766 0.13239083]
 [0.93961804 0.93264309 0.93416429 0.93448232 0.06261829]
 [0.9317587  0.93503128 0.93937038 0.93820633 0.06083424]
 [0.93281162 0.93581504 0.9367483  0.93858253 0.04998854]
 [0.93010429 0.94215895 0.94039653 0.93719067 0.06457

In [15]:
predicted_prices = model.predict(last_60_days)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 910ms/step


In [16]:
print("Scaled Predicted Prices (Open, Close):", predicted_prices)

Scaled Predicted Prices (Open, Close): [[-0.19719674  0.60996646]]


In [17]:
# Rebuild full 5-feature array for inverse transform #
# Format: [Open, High, Low, Close, Volume]
dummy = np.zeros((1, 5))
dummy[0, 0] = predicted_prices[0, 0] # Open dummy[0, 3] = predicted_prices[0, 1] # Close

In [18]:
print("Dummy for Inverse Transform:", dummy)

Dummy for Inverse Transform: [[-0.19719674  0.          0.          0.          0.        ]]


In [19]:
# Inverse transform using same scaler 
predicted_prices_real = scaler.inverse_transform(dummy)[:, [0, 3]] # Get only Open & Close

In [20]:
print(f"📈 Predicted {ticker} Opening Price for Today: ${predicted_prices_real[0][0]:.2f}") 
print(f"📈 Predicted {ticker} Closing Price for Today: ${predicted_prices_real[0][1]:.2f}")

📈 Predicted AAPL Opening Price for Today: $-31.86
📈 Predicted AAPL Closing Price for Today: $20.60


In [22]:
import yfinance as yf 
import datetime
ticker = "AAPL" 
today = datetime.date.today() 
df_today = yf.download(ticker, start=today, end=today + datetime.timedelta(days=1))
print(df_today[['Close']])
print(df_today[['Open']])
df = yf.download("AAPL", period="5d") # last 5 days print(df)

$AAPL: possibly delisted; no price data found  (1d 2026-01-12 -> 2026-01-13)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: possibly delisted; no price data found  (1d 2026-01-12 -> 2026-01-13)


Empty DataFrame
Columns: [(Close, AAPL)]
Index: []
Empty DataFrame
Columns: [(Open, AAPL)]
Index: []


[*********************100%***********************]  1 of 1 completed


In [23]:
 # Step 7: Predict Last 5 Actual Days (Open & Close)
lookback_days = 5 
predicted_prices = []
actual_prices = [] 
for i in range(lookback_days): # Use last 60 days before the day we want to predict 
    start_idx = -(lookback_days + time_step - i) 
    end_idx = -(lookback_days - i) 
    last_seq = scaled_data[start_idx:end_idx] 
    last_seq = last_seq.reshape(1, time_step, X.shape[2]) # Predict Open & Close 
    pred = model.predict(last_seq, verbose=0) # Convert prediction back to original scale 
    dummy = np.zeros((1, 5))
    dummy[0, 0] = pred[0, 0] # Open 
    dummy[0, 3] = pred[0, 1] # Close 
    real_pred = scaler.inverse_transform(dummy)[:, [0, 3]] 
    predicted_prices.append(real_pred[0]) # Get actual Open & Close 
    actual = df[['Open', 'Close']].values[-lookback_days + i]
    actual_prices.append(actual) # Put results into DataFrame 
result_df = pd.DataFrame({ "Actual_Open": [a[0] for a in actual_prices], 
                           "Pred_Open": [p[0] for p in predicted_prices], 
                           "Actual_Close": [a[1] for a in actual_prices], 
                           "Pred_Close": [p[1] for p in predicted_prices] })
print("\n📊 Last 5 Days: Actual vs Predicted")
print(result_df)


📊 Last 5 Days: Actual vs Predicted
   Actual_Open  Pred_Open  Actual_Close  Pred_Close
0   270.640015 -34.505375    267.260010  188.793866
1   267.000000 -34.276565    262.359985  188.189012
2   263.200012 -33.749639    260.329987  186.985509
3   257.019989 -33.072416    259.040009  185.531810
4   259.079987 -32.292112    259.369995  183.833202
